In [ ]:
# ============================================
# TONE GENERATOR + PDF EXPLAINER (COLAB READY)
# ============================================
#
# This script generates customizable audio tones (pure sine waves or
# binaural beats) and creates a companion PDF explaining the math.
#
# USAGE: Run all cells in Google Colab. Audio plays inline and files
# auto-download. Works in Jupyter too, but without auto-download.
#

# CHANGELOG (QA Revision):

# - Added MASTER_VOLUME validation (must be 0.0–1.0)

# - Fixed edge case when all tone toggles are disabled

# - Improved normalize_stereo efficiency

# - Explicit matplotlib backend for headless environments

# - Added configuration summary at end of run

# - Clarified in-place array modification behavior

# - Removed redundant imports

# ============================================

# If running in Colab, install dependencies once per runtime.

# The -q flag suppresses verbose output.

# !pip install -q reportlab matplotlib scipy

import numpy as np
import matplotlib
matplotlib.use('Agg')  # Explicit backend for headless/Colab environments
import matplotlib.pyplot as plt
from scipy.io.wavfile import write as wav_write
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.lib.utils import ImageReader
from datetime import datetime
import io
import sys

# Detect Colab environment for auto-download feature.

# This check runs once at import time.

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    colab_files = None  # Avoid NameError later

from IPython.display import Audio, display, IFrame

# ==============================

# 1. USER CONTROLS

# ==============================

# Adjust these parameters to customize your output.

DURATION_SEC = 120        # Length of audio in seconds
SAMPLE_RATE = 44100       # Samples per second (CD quality = 44100)
MASTER_VOLUME = 0.25      # Peak amplitude after mixing (0.0–1.0)
APPLY_FADE = True         # Fade in/out to prevent clicks at start/end

# — Tone Selection —

# Enable one or more of these to include in the mix.

# Left/Right play only in that ear; Center plays in both.

USE_LEFT_TONE   = False    # Left ear only
USE_RIGHT_TONE  = False    # Right ear only
USE_CENTER_TONE = True     # Both ears equally

# — Frequencies (Hz) —

LEFT_FREQ_HZ   = 528
RIGHT_FREQ_HZ  = 432
CENTER_FREQ_HZ = 528

# — Binaural Beat Mode —

# When enabled, overrides the individual tone settings above.

# Creates a carrier tone with slightly different frequencies in each ear,

# producing an auditory "beat" at the difference frequency.

USE_BINAURAL_BEAT = False
BEAT_HZ    = 3.0           # Perceived beat frequency (2–4 Hz = delta/theta range)
CARRIER_HZ = 200.0         # Base frequency heard in both ears (150–400 Hz typical)

# — Output Filenames —

# Timestamped to avoid overwriting previous runs.

STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
WAV_FILENAME = f"tone_mix_{STAMP}.wav"
PDF_FILENAME = f"tone_info_{STAMP}.pdf"

# ==============================

# 2. INPUT VALIDATION

# ==============================

def validate_config():
    """
    Check user configuration for common errors.
    Raises ValueError with a descriptive message if problems are found.
    Returns a dict with warnings (non-fatal issues) if any.
    """
    warnings = []

    # MASTER_VOLUME must be in valid range
    if not (0.0 <= MASTER_VOLUME <= 1.0):
        raise ValueError(
            f"MASTER_VOLUME must be between 0.0 and 1.0, got {MASTER_VOLUME}. "
            "Values > 1.0 would cause clipping distortion."
        )

    # Check if any audio will actually be generated
    if not USE_BINAURAL_BEAT:
        if not (USE_LEFT_TONE or USE_RIGHT_TONE or USE_CENTER_TONE):
            warnings.append(
                "WARNING: All tone toggles are OFF and binaural mode is disabled. "
                "The output will be silence."
            )

    # Sanity check on frequencies
    for name, freq in [("LEFT_FREQ_HZ", LEFT_FREQ_HZ),
                       ("RIGHT_FREQ_HZ", RIGHT_FREQ_HZ),
                       ("CENTER_FREQ_HZ", CENTER_FREQ_HZ),
                       ("CARRIER_HZ", CARRIER_HZ)]:
        if freq <= 0:
            raise ValueError(f"{name} must be positive, got {freq}")
        if freq > SAMPLE_RATE / 2:
            warnings.append(
                f"WARNING: {name}={freq} Hz exceeds Nyquist frequency "
                f"({SAMPLE_RATE/2} Hz). Aliasing will occur."
            )

    # Duration check
    if DURATION_SEC <= 0:
        raise ValueError(f"DURATION_SEC must be positive, got {DURATION_SEC}")

    return {"warnings": warnings}

# ==============================

# 3. CORE AUDIO FUNCTIONS

# ==============================

def sine_wave(freq_hz, duration, sr):
    """
    Generate a pure sine wave at the specified frequency.

    Parameters:
        freq_hz  : Frequency in Hertz (cycles per second)
        duration : Length in seconds
        sr       : Sample rate (samples per second)

    Returns:
        NumPy array of float64 samples in range [-1.0, 1.0]

    Notes:
        - Uses endpoint=False in linspace to prevent discontinuity if looped.
          With endpoint=True, the last sample would be at t=duration, which
          for a periodic signal creates a duplicate of the first sample.
    """
    n_samples = int(sr * duration)
    t = np.linspace(0, duration, n_samples, endpoint=False)
    return np.sin(2 * np.pi * freq_hz * t)

def apply_fade_io(wave, sr, fade_ms=20):
    """
    Apply fade-in and fade-out to prevent audible clicks.

    This function creates a COPY of the input array to avoid
    unexpected side effects from in-place modification.

    Parameters:
        wave    : 1D NumPy array of audio samples
        sr      : Sample rate
        fade_ms : Duration of fade in milliseconds (default 20ms)

    Returns:
        New array with fades applied
    """
    # Work on a copy to prevent modifying the original
    result = wave.copy()
    n = len(result)
    k = int(sr * (fade_ms / 1000.0))

    # Skip if fade would be too long or array too short
    if k == 0 or k * 2 >= n:
        return result

    # Linear fade envelopes
    fade_in = np.linspace(0.0, 1.0, k)
    fade_out = np.linspace(1.0, 0.0, k)

    result[:k]  *= fade_in
    result[-k:] *= fade_out

    return result

def normalize_stereo(left, right, target_peak):
    """
    Scale stereo audio so the maximum absolute value equals target_peak.

    Parameters:
        left, right : 1D NumPy arrays (same length)
        target_peak : Desired peak amplitude (typically <= 1.0)

    Returns:
        Tuple of (scaled_left, scaled_right) as new arrays
    """
    # Find the true peak across both channels efficiently.
    # Using separate max() calls avoids creating an intermediate list.
    peak_left = np.abs(left).max()
    peak_right = np.abs(right).max()
    peak = max(peak_left, peak_right, 1e-12)  # 1e-12 prevents division by zero

    scale_factor = target_peak / peak
    return left * scale_factor, right * scale_factor

def build_stereo_mix(duration, sr,
                     use_left, f_left,
                     use_right, f_right,
                     use_center, f_center,
                     use_binaural=False, beat_hz=3.0, carrier_hz=200.0,
                     volume=0.25, apply_fade_flag=True):
    """
    Construct the final stereo audio mix based on configuration.

    Parameters:
        duration       : Audio length in seconds
        sr             : Sample rate
        use_left/right/center : Booleans to enable each tone type
        f_left/right/center   : Frequencies for each tone
        use_binaural   : If True, ignores other tones and creates binaural beat
        beat_hz        : Binaural beat frequency (perceived pulsing rate)
        carrier_hz     : Base frequency for binaural beat
        volume         : Target peak amplitude after normalization
        apply_fade_flag: Whether to apply fade in/out

    Returns:
        2D NumPy array of shape (n_samples, 2) with dtype float32
    """
    n = int(sr * duration)
    left  = np.zeros(n, dtype=np.float64)  # Use float64 for intermediate math
    right = np.zeros(n, dtype=np.float64)

    if use_binaural:
        # Binaural beat: slightly different frequencies in each ear.
        # The brain perceives a "beat" at the difference frequency.
        # Example: 200 Hz left, 203 Hz right → 3 Hz perceived beat
        fl = max(0.1, carrier_hz - beat_hz / 2.0)
        fr = max(0.1, carrier_hz + beat_hz / 2.0)
        left  += sine_wave(fl, duration, sr)
        right += sine_wave(fr, duration, sr)
    else:
        # Standard tone mixing
        if use_left:
            left += sine_wave(f_left, duration, sr)
        if use_right:
            right += sine_wave(f_right, duration, sr)
        if use_center:
            # Center tone goes to both channels equally
            center = sine_wave(f_center, duration, sr)
            left  += center
            right += center

    # Normalize to target volume
    left, right = normalize_stereo(left, right, volume)

    # Apply fade to prevent clicks
    if apply_fade_flag:
        left  = apply_fade_io(left, sr)
        right = apply_fade_io(right, sr)

    # Stack into stereo array and convert to float32 for output
    stereo = np.column_stack([left, right]).astype(np.float32)
    return stereo

# ==============================

# 4. PDF GENERATOR

# ==============================

def make_frequency_pdf(mode,
                       sample_rate,
                       amplitude,
                       filename,
                       pure_freq_hz=None,
                       beat_hz=None,
                       carrier_hz=None):
    """
    Create a 1-page PDF explaining the generated tone with a waveform plot.

    The PDF includes:
        - Title describing the tone type
        - General sine wave equation
        - Specific equation(s) for the generated tone
        - Variable definitions
        - Dark-red waveform visualization (first 5ms)

    Parameters:
        mode         : "pure" or "binaural"
        sample_rate  : Fs value to show in equations
        amplitude    : A value (MASTER_VOLUME)
        filename     : Output PDF path
        pure_freq_hz : Frequency for pure tone mode
        beat_hz      : Beat frequency for binaural mode
        carrier_hz   : Carrier frequency for binaural mode
    """
    # Determine which frequency to visualize
    if mode == "pure":
        freq_for_plot = pure_freq_hz
        title = f"{pure_freq_hz} Hz Pure Tone"
    else:
        freq_for_plot = carrier_hz
        title = f"{beat_hz} Hz Binaural Beat (Carrier {carrier_hz} Hz)"

    # --- Create waveform visualization (first 5 ms) ---
    short_ms = 0.005  # 5 milliseconds visible window
    t = np.linspace(0, short_ms, int(sample_rate * short_ms), endpoint=False)
    wave = amplitude * np.sin(2 * np.pi * freq_for_plot * t)

    fig, ax = plt.subplots(figsize=(6, 2))
    ax.plot(t * 1000, wave, color="#8B0000", linewidth=2)  # Dark red color
    ax.set_title(f"{freq_for_plot} Hz Waveform (first 5 ms)", fontsize=12)
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Amplitude")
    ax.grid(True, alpha=0.3)

    # Render to in-memory PNG buffer
    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    waveform_img = ImageReader(buf)

    # --- Build the PDF page ---
    c = canvas.Canvas(filename, pagesize=letter)
    width, height = letter
    y = height - 60

    # Title
    c.setFont("Helvetica-Bold", 18)
    c.drawString(60, y, title)
    y -= 40

    c.setFont("Helvetica", 11)

    # General equation
    c.drawString(60, y, "General discrete-time equation:")
    y -= 18
    c.setFont("Courier", 10)
    c.drawString(80, y, "s[n] = A × sin(2π × f × n / Fs)")
    y -= 28

    # Specific equation(s) for this configuration
    c.setFont("Helvetica", 11)
    if mode == "pure":
        c.drawString(60, y, "Specific equation for this tone:")
        y -= 18
        c.setFont("Courier", 10)
        c.drawString(
            80, y,
            f"s[n] = {amplitude:.2f} × sin(2π × {pure_freq_hz} × n / {sample_rate})"
        )
        y -= 28
    else:
        fl = carrier_hz - beat_hz / 2.0
        fr = carrier_hz + beat_hz / 2.0
        c.drawString(60, y, "Specific equations (left & right ears):")
        y -= 18
        c.setFont("Courier", 10)
        c.drawString(
            80, y,
            f"Left:  s_L[n] = {amplitude:.2f} × sin(2π × {fl:.2f} × n / {sample_rate})"
        )
        y -= 18
        c.drawString(
            80, y,
            f"Right: s_R[n] = {amplitude:.2f} × sin(2π × {fr:.2f} × n / {sample_rate})"
        )
        y -= 28

    # Variable definitions
    c.setFont("Helvetica", 11)
    c.drawString(60, y, "Where:")
    y -= 18
    c.setFont("Helvetica", 10)
    definitions = [
        "A  = amplitude (peak loudness, scale 0–1)",
        "f  = frequency in Hz (cycles per second)",
        "n  = sample index (0, 1, 2, ...)",
        f"Fs = sample rate ({sample_rate} samples/second)"
    ]
    for defn in definitions:
        c.drawString(80, y, defn)
        y -= 16
    y -= 14

    # Waveform image
    c.setFont("Helvetica", 11)
    c.drawString(60, y, "Waveform visualization (first 5 ms):")
    y -= 10
    c.drawImage(waveform_img, 60, y - 120, width=400, height=120)
    y -= 150

    # Footer note
    c.setFont("Helvetica-Oblique", 9)
    c.drawString(
        60, y,
        "Generated digitally using the sampled sine-wave equation above."
    )

    # Timestamp
    y -= 20
    c.setFont("Helvetica", 8)
    c.drawString(60, y, f"Created: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    c.save()

def print_config_summary():
    """Print a human-readable summary of the current configuration."""
    print("\n" + "=" * 50)
    print("CONFIGURATION SUMMARY")
    print("=" * 50)
    print(f"Duration:      {DURATION_SEC} seconds")
    print(f"Sample Rate:   {SAMPLE_RATE} Hz")
    print(f"Master Volume: {MASTER_VOLUME}")
    print(f"Fade In/Out:   {'Yes' if APPLY_FADE else 'No'}")
    print("-" * 50)

    if USE_BINAURAL_BEAT:
        fl = CARRIER_HZ - BEAT_HZ / 2.0
        fr = CARRIER_HZ + BEAT_HZ / 2.0
        print(f"Mode:          Binaural Beat")
        print(f"Carrier:       {CARRIER_HZ} Hz")
        print(f"Beat Freq:     {BEAT_HZ} Hz")
        print(f"Left Ear:      {fl:.2f} Hz")
        print(f"Right Ear:     {fr:.2f} Hz")
    else:
        print(f"Mode:          Standard Tones")
        tones = []
        if USE_LEFT_TONE:
            tones.append(f"Left ({LEFT_FREQ_HZ} Hz)")
        if USE_RIGHT_TONE:
            tones.append(f"Right ({RIGHT_FREQ_HZ} Hz)")
        if USE_CENTER_TONE:
            tones.append(f"Center ({CENTER_FREQ_HZ} Hz)")
        if tones:
            print(f"Active Tones:  {', '.join(tones)}")
        else:
            print(f"Active Tones:  NONE (output will be silence)")

    print("-" * 50)
    print(f"WAV Output:    {WAV_FILENAME}")
    print(f"PDF Output:    {PDF_FILENAME}")
    print("=" * 50 + "\n")

# ==============================

# 5. MAIN EXECUTION

# ==============================

if __name__ == "__main__":
    # 5a. Validate configuration before doing any work
    try:
        validation = validate_config()
        for warning in validation["warnings"]:
            print(warning)
    except ValueError as e:
        print(f"CONFIGURATION ERROR: {e}")
        sys.exit(1)

    # 5b. Show configuration summary
    print_config_summary()

    # 5c. Build the stereo audio
    print("Generating audio...")
    audio = build_stereo_mix(
        DURATION_SEC, SAMPLE_RATE,
        USE_LEFT_TONE, LEFT_FREQ_HZ,
        USE_RIGHT_TONE, RIGHT_FREQ_HZ,
        USE_CENTER_TONE, CENTER_FREQ_HZ,
        use_binaural=USE_BINAURAL_BEAT,
        beat_hz=BEAT_HZ,
        carrier_hz=CARRIER_HZ,
        volume=MASTER_VOLUME,
        apply_fade_flag=APPLY_FADE
    )

    # 5d. Export WAV file
    # Convert float32 [-1.0, 1.0] to int16 [-32767, 32767]
    wav_int16 = np.int16(np.clip(audio, -1.0, 1.0) * 32767)
    wav_write(WAV_FILENAME, SAMPLE_RATE, wav_int16)
    print(f"✓ WAV export complete: {WAV_FILENAME}")

    # 5e. Inline audio playback (Colab/Jupyter)
    try:
        display(Audio(WAV_FILENAME))
    except Exception as e:
        print(f"  (Inline audio playback unavailable: {e})")

    # 5f. Auto-download WAV (Colab only)
    if IN_COLAB and colab_files:
        try:
            colab_files.download(WAV_FILENAME)
        except Exception as e:
            print(f"  WAV auto-download failed; use file browser. Error: {e}")

    # 5g. Determine what to document in the PDF
    if USE_BINAURAL_BEAT:
        pdf_mode = "binaural"
        pure_freq = None
    else:
        pdf_mode = "pure"
        # Pick the active tone frequency for documentation.
        # Priority: center > left > right (arbitrary but consistent)
        if USE_CENTER_TONE:
            pure_freq = CENTER_FREQ_HZ
        elif USE_LEFT_TONE:
            pure_freq = LEFT_FREQ_HZ
        elif USE_RIGHT_TONE:
            pure_freq = RIGHT_FREQ_HZ
        else:
            # All tones disabled - still create PDF but note this edge case
            pure_freq = CENTER_FREQ_HZ
            print("  Note: PDF will document CENTER_FREQ_HZ even though no tones are active.")

    # 5h. Create PDF
    print("Generating PDF...")
    if pdf_mode == "pure":
        make_frequency_pdf(
            mode="pure",
            sample_rate=SAMPLE_RATE,
            amplitude=MASTER_VOLUME,
            filename=PDF_FILENAME,
            pure_freq_hz=pure_freq
        )
    else:
        make_frequency_pdf(
            mode="binaural",
            sample_rate=SAMPLE_RATE,
            amplitude=MASTER_VOLUME,
            filename=PDF_FILENAME,
            beat_hz=BEAT_HZ,
            carrier_hz=CARRIER_HZ
        )
    print(f"✓ PDF export complete: {PDF_FILENAME}")

    # 5i. Inline PDF display
    # Note: IFrame often fails in Colab's sandboxed environment.
    # The PDF will still be downloadable from the file browser.
    try:
        display(IFrame(PDF_FILENAME, width=600, height=800))
    except Exception as e:
        print(f"  (Inline PDF display unavailable: {e})")
        print("  You can view the PDF from the file browser in the left panel.")

    # 5j. Auto-download PDF (Colab only)
    if IN_COLAB and colab_files:
        try:
            colab_files.download(PDF_FILENAME)
        except Exception as e:
            print(f"  PDF auto-download failed; use file browser. Error: {e}")

    print("\n✓ Done!")